# 02 - Ingestão de PDFs (Estratégia Híbrida)

Este notebook processa PDFs usando estratégia inteligente:

**Estratégia Híbrida:**
- Páginas com figuras/tabelas/gráficos → Imagem PNG
- Páginas apenas com texto → Texto extraído
- Páginas de referências → Ignoradas

**Benefícios:**
- Menor consumo de API (menos imagens)
- Mantém precisão visual onde importa
- Ignora conteúdo irrelevante (referências)

## 1. Setup

In [1]:
import sys
import json
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
SYSTEM_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = SYSTEM_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import paths, load_mapping, extraction_config
from pdf_vision import (
    get_pdf_info, 
    check_pdf_quality, 
    analyze_pdf_pages,
    extract_hybrid
)

print(f"✅ Módulos carregados")
print(f"   DPI para imagens: {extraction_config.IMAGE_DPI}")
print(f"   Estratégia: Híbrida (texto + imagens seletivas)")

✅ Módulos carregados
   DPI para imagens: 300
   Estratégia: Híbrida (texto + imagens seletivas)


## 2. Carregar Mapping

In [2]:
mapping = load_mapping(paths.MAPPING_CSV)
print(f"📚 {len(mapping)} artigos no mapping")

# Verificar quais já foram ingeridos (têm arquivo hybrid.json)
ingested = []
pending = []

for article in mapping:
    artigo_id = article["ArtigoID"]
    hybrid_file = paths.WORK / artigo_id / "hybrid.json"
    if hybrid_file.exists():
        ingested.append(article)
    else:
        pending.append(article)

print(f"\n📊 Status:")
print(f"   Já ingeridos: {len(ingested)}")
print(f"   Pendentes: {len(pending)}")

📚 38 artigos no mapping

📊 Status:
   Já ingeridos: 38
   Pendentes: 0


## 3. Função de Ingestão Híbrida

In [3]:
def ingest_article_hybrid(artigo_id: str, arquivo: str, force: bool = False) -> dict:
    """
    Ingere um artigo usando estratégia híbrida.
    
    Args:
        artigo_id: ID do artigo (ART_001)
        arquivo: Nome do arquivo PDF
        force: Se True, reprocessa mesmo se já existir
    
    Returns:
        Dict com status e métricas
    """
    pdf_path = paths.ARTICLES / arquivo
    work_dir = paths.WORK / artigo_id
    pages_dir = work_dir / "pages"
    hybrid_file = work_dir / "hybrid.json"
    
    # Verificar se já foi ingerido
    if not force and hybrid_file.exists():
        with open(hybrid_file, "r", encoding="utf-8") as f:
            cached = json.load(f)
        stats = cached.get("stats", {})
        return {
            "status": "cached",
            "stats": stats,
            "message": f"Já ingerido (texto:{stats.get('text_pages',0)}, img:{stats.get('image_pages',0)}, skip:{stats.get('skipped_pages',0)})"
        }
    
    # Verificar se PDF existe
    if not pdf_path.exists():
        return {
            "status": "error",
            "stats": {},
            "message": f"PDF não encontrado: {arquivo}"
        }
    
    # Info do PDF
    info = get_pdf_info(pdf_path)
    quality = check_pdf_quality(pdf_path)
    
    print(f"   📄 {info['num_pages']} páginas, {info['file_size_mb']:.1f}MB")
    print(f"   📊 Qualidade texto: {quality['quality_score']}")
    
    # Analisar páginas
    print(f"   🔍 Analisando páginas...")
    analysis = analyze_pdf_pages(pdf_path)
    
    # Contar por estratégia (text_partial conta como texto)
    text_count = len([a for a in analysis if a["strategy"] in ["text", "text_partial"]])
    image_count = len([a for a in analysis if a["strategy"] == "image"])
    skip_count = len([a for a in analysis if a["strategy"] == "skip"])
    
    print(f"   📋 Análise: {text_count} texto, {image_count} imagem, {skip_count} ignoradas")
    
    # Ícones para cada estratégia
    icons = {
        "text": "📝",
        "text_partial": "📝✂️",
        "image": "🖼️",
        "skip": "⏭️"
    }
    
    # Mostrar detalhes da análise
    for a in analysis:
        icon = icons.get(a["strategy"], "❓")
        print(f"      {icon} Pág {a['page_num']}: {a['reason']}")
    
    # Extrair conteúdo híbrido
    print(f"\n   🔄 Extraindo conteúdo...")
    result = extract_hybrid(pdf_path, pages_dir, dpi=extraction_config.IMAGE_DPI)
    
    # Salvar resultado (sem o conteúdo de texto, só metadados)
    hybrid_meta = {
        "artigo_id": artigo_id,
        "arquivo": arquivo,
        "analysis": result["analysis"],
        "stats": result["stats"],
        "pages_info": [
            {
                "page_num": p["page_num"],
                "type": p["type"],
                "path": str(p["path"]) if p["type"] == "image" else None
            }
            for p in result["pages"]
        ]
    }
    
    work_dir.mkdir(parents=True, exist_ok=True)
    with open(hybrid_file, "w", encoding="utf-8") as f:
        json.dump(hybrid_meta, f, indent=2, ensure_ascii=False)
    
    # Salvar textos extraídos separadamente
    texts_file = work_dir / "texts.json"
    texts_data = {
        p["page_num"]: p["content"]
        for p in result["pages"]
        if p["type"] == "text"
    }
    with open(texts_file, "w", encoding="utf-8") as f:
        json.dump(texts_data, f, indent=2, ensure_ascii=False)
    
    stats = result["stats"]
    return {
        "status": "success",
        "stats": stats,
        "message": f"texto:{stats['text_pages']}, img:{stats['image_pages']}, skip:{stats['skipped_pages']}"
    }

## 4. Ingerir Artigo Teste (Piloto)

In [ ]:
# Selecionar primeiro artigo para teste
test_article = mapping[0]

print(f"🧪 Artigo teste:")
print(f"   ID: {test_article['ArtigoID']}")
print(f"   Arquivo: {test_article['Arquivo'][:60]}...")

# Executar ingestão híbrida
print(f"\n🔄 Ingerindo com estratégia híbrida...")
result = ingest_article_hybrid(
    test_article["ArtigoID"], 
    test_article["Arquivo"],
    force=True  # Forçar para ver a análise
)

print(f"\n✅ Resultado: {result['status']}")
print(f"   {result['message']}")

In [ ]:
# Verificar arquivos gerados
work_dir = paths.WORK / test_article["ArtigoID"]

print(f"📁 Arquivos em {work_dir}:")

# hybrid.json
hybrid_file = work_dir / "hybrid.json"
if hybrid_file.exists():
    print(f"   ✅ hybrid.json ({hybrid_file.stat().st_size / 1024:.1f} KB)")

# texts.json
texts_file = work_dir / "texts.json"
if texts_file.exists():
    print(f"   ✅ texts.json ({texts_file.stat().st_size / 1024:.1f} KB)")

# Imagens
pages_dir = work_dir / "pages"
if pages_dir.exists():
    images = sorted(pages_dir.glob("*.png"))
    print(f"   ✅ {len(images)} imagens PNG:")
    for img in images[:5]:
        size_kb = img.stat().st_size / 1024
        print(f"      {img.name} ({size_kb:.0f} KB)")
    if len(images) > 5:
        print(f"      ... e mais {len(images) - 5} imagens")

## 5. Visualizar Página de Exemplo (Imagem)

In [ ]:
from IPython.display import Image, display

pages_dir = paths.WORK / test_article["ArtigoID"] / "pages"
images = sorted(pages_dir.glob("*.png"))

if images:
    print(f"📄 Preview: {images[0].name}")
    display(Image(filename=str(images[0]), width=600))
else:
    print("Nenhuma imagem gerada (todas as páginas são texto)")

## 6. Ingerir Todos os Artigos (Batch)

⚠️ **Execute apenas quando pronto para processar todos os PDFs.**

In [ ]:
# Switch de segurança
RUN_BATCH = True  # mude para True quando estiver pronto

if not RUN_BATCH:
    print("🛑 Batch não executado (RUN_BATCH=False).")
    print("   Quando estiver pronto, defina RUN_BATCH=True e reexecute esta célula.")
else:
    from tqdm.notebook import tqdm

    print(f"🚀 Ingerindo {len(pending)} artigos pendentes...\n")

    results = []
    for article in tqdm(pending, desc="Ingerindo"):
        artigo_id = article["ArtigoID"]
        arquivo = article["Arquivo"]

        print(f"\n{artigo_id}: {arquivo[:40]}...")
        try:
            result = ingest_article_hybrid(artigo_id, arquivo)
            results.append({"id": artigo_id, **result})
        except Exception as e:
            print(f"   ❌ Erro: {e}")
            results.append({"id": artigo_id, "status": "error", "message": str(e)})

    # Resumo
    success = len([r for r in results if r["status"] == "success"])
    cached = len([r for r in results if r["status"] == "cached"])
    errors = len([r for r in results if r["status"] == "error"])

    print(f"\n" + "="*50)
    print(f"✅ Sucesso: {success}")
    print(f"⏭️ Cache: {cached}")
    print(f"❌ Erros: {errors}")

## 7. Próximo Passo

In [ ]:
# Verificar status final
total_ingested = 0
for article in mapping:
    hybrid_file = paths.WORK / article["ArtigoID"] / "hybrid.json"
    if hybrid_file.exists():
        total_ingested += 1

print(f"📊 Status de Ingestão:")
print(f"   Total: {len(mapping)}")
print(f"   Ingeridos: {total_ingested}")
print(f"   Pendentes: {len(mapping) - total_ingested}")

if total_ingested > 0:
    print(f"\n✅ Pronto para 03_Extracao_LLM.ipynb")
else:
    print(f"\n⚠️ Ingira pelo menos 1 artigo antes de continuar")

In [ ]:
article = mapping[1]  # ART_002 (se o mapping estiver na ordem)
ingest_article_hybrid(article["ArtigoID"], article["Arquivo"], force=False)

In [ ]:
TARGET_IDS = {"ART_003","ART_004","ART_005","ART_006","ART_007"}  # exemplo
to_ingest = [a for a in pending if a["ArtigoID"] in TARGET_IDS]

print("Vai ingerir:", [a["ArtigoID"] for a in to_ingest])

for a in to_ingest:
    print(f"\n▶️ Ingerindo {a['ArtigoID']} - {a['Arquivo'][:60]}...")
    r = ingest_article_hybrid(a["ArtigoID"], a["Arquivo"], force=False)
    print(r)